In [49]:
import numpy as np
import tifffile
from readlif.reader import LifFile
import os
import matplotlib.pyplot as plt
import CellPatchExtraction_package.CellPatchExtraction.src.extraction as ex
from cellplot_package.cellplot.segmentation import rand_col_seg, contoure_seg
import cv2
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from pathlib import Path
import pandas as pd
import json
import cv2
from skimage.feature import blob_log

images_folder = Path("/home/simon_g/isilon_images_mnt/10_MetaSystems/MetaSystemsData/_simon/src/sören_foci/images")
annotations_folder = Path("/home/simon_g/isilon_images_mnt/10_MetaSystems/MetaSystemsData/_simon/src/sören_foci/annotations")
raw_folder = Path('/home/simon_g/isilon_images_mnt/10_MetaSystems/MetaSystemsData/_simon/src/sören_foci/raw_images')

experiments = list(images_folder.glob('*'))
experiments

for experiment in experiments:
    
    images = list(experiment.glob('*image*.png'))
    annotations = pd.read_csv(annotations_folder / (experiment.stem + '.csv'))
    
    print(experiment.stem)
    if experiment.stem in ['CM_JUN_TEAD', 'CM_TAZ_TEAD']:
        pass
        # continue

    experiment_df = dict(spots=[], classes=[])
    for image_file in images:
        
        raw_image = raw_folder / experiment.stem / f'raw_{image_file.stem.split("_")[1]}.tif'
        raw_image = tifffile.imread(raw_image).transpose(1,2,3,0)
        
        #foci = raw_image[3].sum(axis=-1)/255        
        foci = raw_image[3, ..., 0]/255

        blurred = cv2.GaussianBlur(foci, ksize=(11, 11), sigmaX=1)
        enhanced_sub = foci - blurred
        enhanced = np.clip(enhanced_sub, 0, None)
        
        log_blobs = blob_log(enhanced, min_sigma=0.5, max_sigma=5, num_sigma=10, threshold=0.1)[:, :2].astype(int)
        
        # Normalize for visualization
        enhanced = (enhanced - enhanced.min()) / (enhanced.max() - enhanced.min() + 1e-8)

        curr_annotations = annotations[annotations.img.str.contains(image_file.stem)]
        
        mask = str(image_file.name).replace('image', 'masks')
        mask = mask.replace('png', 'tif')

        image = plt.imread(image_file)[:512, :512, :3]
        mask = tifffile.imread(image_file.parent / mask)
        
        dilated_mask = np.copy(mask)
        dilation_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))  # tune size as needed

        for mask_id in np.unique(mask):
            if mask_id == 0:
                continue

            cell_mask = (mask == mask_id).astype(np.uint8)
            dilated = cv2.dilate(cell_mask, dilation_kernel)

            # Only allow dilation into background (to prevent overwrite)
            new_pixels = (dilated == 1) & (mask == 0)
            dilated_mask[new_pixels] = mask_id
        
        image_annotations = pd.DataFrame(json.loads(curr_annotations['kp-1'].item()))
        #print(image_annotations)
        
        x = (image_annotations['x']/100 * image_annotations['original_width']).to_numpy()
        y = (image_annotations['y']/100 * image_annotations['original_height']).to_numpy()
        labels = np.array([label[0] for label in image_annotations['keypointlabels']])
        
        if any(x > 512):
            y = y[x > 512]
            x = x[x > 512]
            x -= 512
            
        # fig, ax = plt.subplots(1, 3, figsize=(10, 5))
        # ax[0].imshow(image)
        # ax[1].imshow(mask)
        # ax[2].imshow(dilated_mask)
        # ax[0].scatter(x, y, s=10, marker='x', c='red')
        # plt.show()
        
        # fig, ax = plt.subplots(1, 3, figsize=(20, 10))
        # ax[0].imshow(foci, vmin=0, vmax=0.2)
        # ax[1].imshow(enhanced, vmin=0, vmax=0.2)
        # ax[2].imshow(mask)
        # ax[2].scatter(log_blobs[:, 1], log_blobs[:, 0], s=10, marker='x', c='r')
        # plt.show()
        
        
        blob_values = []
        for blob in log_blobs:
            blob_values.append(dilated_mask[blob[0], blob[1]])
           
        log_blobs = np.concatenate([log_blobs, np.atleast_2d(blob_values).T], axis=1)
        blob_df = pd.DataFrame(log_blobs, columns=['X', 'Y', 'mask_idx'])
        
        mask_annotations = []
        for x, y, labels in zip(x.astype(int), y.astype(int), labels):
            mask_annotations.append([dilated_mask[y, x], labels])
        
        mask_annotations = np.array(mask_annotations, dtype=object)
        mask_df = pd.DataFrame(mask_annotations, columns=['mask_idx', 'label'])
        
        # Step 1: Count number of blobs per mask
        blob_counts = blob_df.groupby('mask_idx').size().reset_index(name='n_spots')

        # Step 2: Merge with cell label annotations
        merged = pd.merge(blob_counts, mask_df, on='mask_idx', how='inner')  # Only annotated cells

        experiment_df['spots'].extend(merged.n_spots)
        experiment_df['classes'].extend(merged.label)

    experiment_df = pd.DataFrame(experiment_df)
    summary = experiment_df.groupby('classes')['spots'].agg(['mean', 'std', 'count']).round(2)
    print(f"\nSummary for experiment: {experiment.stem}")
    print(summary)

CM_JUN_TEAD

Summary for experiment: CM_JUN_TEAD
         mean   std  count
classes                   
ADR      1.24  0.52     25
MES      5.33  9.08      9
OTHER    1.41  0.80     37
CM_TAZ_TEAD

Summary for experiment: CM_TAZ_TEAD
         mean   std  count
classes                   
ADR      1.52  0.98    115
MES      8.47  9.50     51
OTHER    2.41  2.32    128
CM_YAP_TEAD

Summary for experiment: CM_YAP_TEAD
         mean   std  count
classes                   
ADR      1.21  0.43     14
MES      4.73  3.15     22
OTHER    1.62  0.87     55
